# Equality study — aggregation

Reads the shards banked on Drive by `2a` / `2b` / `2c` and writes the CSVs that
`3. Results Data Analysis.ipynb` consumes. It runs no simulations.

Separating aggregation from computation is the point. A run notebook is a
multi-day job that ends by releasing its runtime, so refreshing a summary
through it means either re-entering a study you did not want to advance or
waiting on one that is still going.

## Built for a study that is not finished

A partial study is the normal case here, not an error case — no Colab session
survives a full run, so at any given moment some arm is short. Three properties
follow, and each is exercised by `unit_tests/test_equality_study_aggregation.py`:

- **It aggregates what is banked.** An arm reports the variants on disk. No
  target is consulted, nothing waits for a goal to be reached, and 342 of 500
  variants summarise to 342 rows rather than to an error.
- **It resumes.** Each arm caches its own summary beside its shards, so a later
  session reads only the variants banked since. Aggregation itself can run out
  of time and pick up where it stopped. This is sound because every aggregate is
  computed *within* one variant — a subset summary is identical to the same rows
  of a whole-arm summary, so partial passes concatenate without qualification.
- **One bad arm costs one arm.** A read that fails midway through an option is
  caught, recorded, and the remaining arms still aggregate. Twenty minutes of
  Drive reads are not forfeited to a transient I/O error on the last shard.

What it will not do is let a partial run pass for a complete one. Every arm's
banked-versus-target count is written to `completeness.csv` beside the files
notebook 3 opens, and any arm short of its target is named in the output.

## What it writes

Into each option's own `<slug>/<RUN_TAG>/` directory, beside its shards:

| File | Rows |
|---|---|
| `variant_summary.csv` | one per variant, every arm of that option |
| `variant_summary_<network>.csv` | the same rows, split by network |
| `<arm>/arm_summary_cache.*` | the resume cache — one per arm, safe to delete |

Into `<STUDY_ROOT>/notebook3_inputs/<RUN_TAG>/`, one folder to download whole:

| File | Rows |
|---|---|
| `<slug>_summary.csv` | identical to that option's `variant_summary.csv`, under the name notebook 3 opens |
| `completeness.csv` | banked vs target per arm — how much of each file is actually there |

And into the `equality_study/` root, across all options:

| File | Rows |
|---|---|
| `all_options_variant_summary_<RUN_TAG>.csv` | every variant of every option, with an `option` column |
| `all_options_overview_<RUN_TAG>.csv` | one per (option, network, arm): variant count, completeness, mean outcomes |

## Handing off to notebook 3

Notebook 3 opens `results/<slug>_summary.csv` from the **repo**, not from Drive,
and does its own `df[df["network"] == ...]` split — so it wants the combined
file, one per option, under that exact name. That is what
`notebook3_inputs/<RUN_TAG>/` holds: download the folder and drop its contents
into the repo's `results/`. No rename, no reshape.

The column contract is asserted rather than assumed — see the check cell near
the end. It compares against the 19 columns of the `option1` and `option3` files
already in `results/`, so a change to `summarise_arm` that would break notebook 3
fails here instead of three notebooks later.

# Setup

In [ ]:
# ── Environment switch + parameters (single source of truth) ────────────────
# RUNNING_LOCALLY=True  → laptop, run from the repo clone (no clone/pip/Drive).
# RUNNING_LOCALLY=False → Colab: force-fresh clone, pip install, mount Drive.
import os, sys, subprocess, shutil, traceback
from pathlib import Path

RUNNING_LOCALLY = False
RUN_TAG         = 'full'      # which tier to aggregate: 'full' or 'smoke'
RESET_CACHE     = False       # True → ignore every arm cache and re-read all shards

# This notebook never writes a shard, never touches a config stamp, and never
# deletes anything. What it does write beside the shards is each arm's summary
# cache — the thing that makes a second aggregation session cheap. Deleting a
# cache costs only the re-read; RESET_CACHE=True does it for every arm at once,
# which is the move after changing summarise_arm.
#
# 'smoke' aggregates the 3-variant plumbing runs. Those numbers measure nothing;
# aggregate them only to check this notebook itself works. RUN_TAG is part of
# every path written above the smoke/full split, so a smoke pass cannot
# overwrite a full one.

In [ ]:
# Force-fresh clone + deps (Colab only). Uses subprocess/os.chdir rather than
# !/% magics so the RUNNING_LOCALLY guard actually holds — a line magic fires
# regardless of the surrounding `if` (NOTEBOOK_WRITING_SKILL §5–6).
if not RUNNING_LOCALLY:
    # Anchor on an absolute base, never the cwd: resolving the repo name
    # relative to the cwd makes a second run clone INSIDE the first clone.
    COLAB_BASE = Path('/content')
    os.chdir(COLAB_BASE)
    repo_dir = COLAB_BASE / 'e_network_inequality'
    if repo_dir.exists():
        shutil.rmtree(repo_dir)
    subprocess.run(['git', 'clone', '-b', 'main',
                    'https://github.com/IgnacioOQ/e_network_inequality'], check=True)
    subprocess.run(['pip', 'install', '-q', 'dill'], check=True)
    os.chdir(repo_dir)
else:
    # This notebook lives at the repo root; walk up to the dir containing model/.
    PROJECT_ROOT = Path.cwd()
    while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'model').is_dir():
        PROJECT_ROOT = PROJECT_ROOT.parent
    os.chdir(PROJECT_ROOT)

sys.path.insert(0, os.getcwd())
REPO_ROOT = Path.cwd()
print("Working dir:", REPO_ROOT)

In [ ]:
# Explicit imports — no wildcards from external libraries (§4).
import json

import pandas as pd

# Internal package. Note what is NOT imported: nothing that builds or runs a
# simulation, and no network pickles. Arms are discovered from the directory
# names on disk, so this notebook needs neither the graphs nor a config cell
# agreeing with the run notebooks about how many variants there should be.
#
# read_shard/write_shard are reused for the arm cache rather than plain
# to_csv/read_csv: write_shard renames a temp file into place, so a session
# killed mid-write leaves the previous cache intact instead of a truncated one
# the next session would trust.
from model.equality_study import (
    METHODS,
    OUTCOMES,
    SHARD_EXT,
    completed_variants,
    read_shard,
    summarise_arm,
    write_shard,
)

print("Shard format:", SHARD_EXT)

# Configuration

In [ ]:
# The option trees to aggregate. Each slug must match OPTION_SLUG in that run
# notebook's config cell — the slug IS the directory name on Drive, and it is
# also the stem of the file notebook 3 opens. A typo here reads as "option not
# present" rather than as an error, which is what the inventory cell is for.
OPTIONS = [
    ('option1_literature',       'Option 1 — Following the literature', '2a'),
    ('option2_harder',           'Option 2 — Harder problems',          '2b'),
    ('option3_phase_transition', 'Option 3 — Phase transition',         '2c'),
]

if RUNNING_LOCALLY:
    STUDY_ROOT = Path('results/equality_study')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    STUDY_ROOT = Path('/content/drive/My Drive/Colab Projects/Data Driven ABMs/'
                      'Data Sets/equality_study')
STUDY_ROOT = Path(STUDY_ROOT)

# Everything notebook 3 needs, in one folder to download. RUN_TAG is a path
# component, not just a filename suffix, so a smoke aggregation lands somewhere
# a full one will never be read from.
NOTEBOOK3_DIR = STUDY_ROOT / 'notebook3_inputs' / RUN_TAG
NOTEBOOK3_DIR.mkdir(parents=True, exist_ok=True)

CACHE_NAME = f'arm_summary_cache{SHARD_EXT}'

print("Study root:     ", STUDY_ROOT)
print("Run tag:        ", RUN_TAG)
print("Notebook 3 dir: ", NOTEBOOK3_DIR)


def option_dir(slug):
    """Where one option's shards live. Mirrors RESULTS_DIR in the run notebooks."""
    return STUDY_ROOT / slug / RUN_TAG


def arm_dirs(directory):
    """The (path, network, method) arms under an option directory.

    Discovered from the directory names rather than from a network list, so an
    arm this notebook has never heard of still gets aggregated. Matching on the
    method SUFFIX rather than splitting on '_' is deliberate: network labels are
    free to contain underscores, method names are a closed set.
    """
    if not directory.is_dir():
        return []
    found = []
    for path in sorted(p for p in directory.iterdir() if p.is_dir()):
        method = next((m for m in METHODS if path.name.endswith('_' + m)), None)
        if method is not None:
            found.append((path, path.name[: -len(method) - 1], method))
    return found


def save_csv(df, directory, name):
    path = Path(directory) / name
    df.to_csv(path, index=False)
    print(f"    Saved: {name} ({len(df):,} rows) -> {directory}")
    return path

# Resumable per-arm summaries

In [ ]:
def arm_summary(path, reset=False):
    """One arm's per-variant summary, reading only the shards not already cached.

    Returns ``(summary, n_from_cache, n_read_now)``.

    The cache is what makes a partial study cheap to re-aggregate. Reading an
    arm means opening every shard in it, and at 500 variants x 500 replicates
    that is a quarter-million rows across a Drive mount — so a session that adds
    158 variants should pay for 158, not for 658. Because summarise_arm
    aggregates strictly within a variant, the cached rows and the fresh ones are
    the same rows a single whole-arm pass would produce; the unit tests pin that
    equivalence so it cannot quietly stop being true.

    The cache is keyed on variant indices actually present on disk, not on a
    count or a timestamp. A shard that disappears drops out of the summary
    rather than lingering in it, and a mtime that a Drive mount reports oddly
    cannot make a stale cache look current.
    """
    banked = completed_variants(path)
    cache_path = path / CACHE_NAME

    cached = pd.DataFrame()
    if not reset and cache_path.exists():
        try:
            cached = read_shard(cache_path)
            cached = cached[cached['variant_index'].isin(banked)]
        except Exception as exc:
            # A damaged cache is a performance problem, never a correctness one:
            # drop it and read the shards, which are the only source of truth.
            print(f"    (cache unreadable, re-reading shards: {type(exc).__name__})")
            cached = pd.DataFrame()

    have = set(cached['variant_index']) if len(cached) else set()
    to_read = sorted(banked - have)

    parts = [cached] if len(cached) else []
    if to_read:
        parts.append(summarise_arm(path, variants=to_read))
    parts = [p for p in parts if len(p)]
    if not parts:
        return pd.DataFrame(), 0, 0

    summary = (pd.concat(parts, ignore_index=True)
                 .sort_values('variant_index')
                 .reset_index(drop=True))
    if to_read:
        write_shard(summary, cache_path)
    return summary, len(have), len(to_read)

# What is on disk

In [ ]:
# Cheap by construction: directory listings plus each option's small JSON stamp.
# No shard is opened, so this stays instant across the Drive mount even when the
# study holds millions of rows — and it tells you what the expensive cell below
# is about to read before you pay for it.
rows = []
for slug, title, notebook in OPTIONS:
    directory = option_dir(slug)
    if not directory.is_dir():
        print(f"  {slug}: NOT PRESENT at {directory}")
        continue
    stamp = directory / 'equality_study_config.json'
    config = json.loads(stamp.read_text()) if stamp.exists() else {}
    for path, network, method in arm_dirs(directory):
        banked = len(completed_variants(path))
        target = config.get('n_variants')
        rows.append({
            'option': slug,
            'notebook': notebook,
            'network': network,
            'method': method,
            'variants_banked': banked,
            'variants_target': target,
            'pct': round(100.0 * banked / target, 1) if target else None,
            'cached': (path / CACHE_NAME).exists(),
            'n_runs': config.get('n_runs'),
            'uncertainty': config.get('uncertainty'),
            'window': config.get('window'),
        })

INVENTORY = pd.DataFrame(rows)

# A zero here is a claim about a network filesystem, and Drive has been observed
# reporting an empty listing for a directory holding thousands of shards. Treat
# it as a prompt to look again, not as a finding — and never as grounds to
# re-run an arm that may already be complete.
if len(INVENTORY) and (INVENTORY['variants_banked'] == 0).any():
    empty = INVENTORY[INVENTORY['variants_banked'] == 0]
    print(f"\n  !!! {len(empty)} arm(s) list ZERO banked variants. Absence is not "
          "evidence on a mounted\n      filesystem — confirm with a second listing "
          "before concluding they are unstarted.")

INVENTORY

# Aggregate

In [ ]:
%%time
# The expensive cell, and the only one that opens a shard. Each arm is isolated:
# a read that fails partway through an option is recorded and the remaining arms
# still aggregate, because losing twenty minutes of completed Drive reads to a
# transient error on the last shard is the failure mode worth engineering away.
# Same reasoning as run_study's continue_on_error, which exists for the same
# reason one arm up the pipeline.
all_options, completeness, failures = [], [], []

for slug, title, notebook in OPTIONS:
    directory = option_dir(slug)
    arms = arm_dirs(directory)
    if not arms:
        print(f"\n{title}: SKIPPED — no arm directories under {directory}")
        continue

    stamp = directory / 'equality_study_config.json'
    config = json.loads(stamp.read_text()) if stamp.exists() else {}
    target = config.get('n_variants')

    print(f"\n{title}  ({directory})")
    frames = []
    for path, network, method in arms:
        try:
            summary, from_cache, read_now = arm_summary(path, reset=RESET_CACHE)
        except Exception as exc:
            failures.append({'option': slug, 'network': network, 'method': method,
                             'error': f'{type(exc).__name__}: {exc}',
                             'traceback': traceback.format_exc()})
            print(f"  !!! {path.name}: FAILED — isolated, continuing. "
                  f"{type(exc).__name__}: {exc}")
            continue

        banked = len(summary)
        note = f"{from_cache:,} cached + {read_now:,} read" if from_cache else f"{read_now:,} read"
        short = '' if target is None or banked >= target else f"  <-- PARTIAL, {target - banked:,} short"
        print(f"  {path.name}: {banked:,} variants ({note}){short}")

        completeness.append({
            'option': slug, 'notebook': notebook, 'network': network, 'method': method,
            'variants_summarised': banked, 'variants_target': target,
            'pct': round(100.0 * banked / target, 1) if target else None,
            'complete': None if target is None else banked >= target,
        })
        if len(summary):
            frames.append(summary)

    if not frames:
        # Write NOTHING rather than an empty CSV. An empty file is worse than a
        # missing one: it reads downstream as a measured zero, while a missing
        # file is unambiguously "not aggregated". See the listing caveat above.
        print(f"  !!! no shards read for {slug} — writing nothing. Re-check the "
              "listing before\n      concluding this option is empty.")
        continue

    summary = pd.concat(frames, ignore_index=True)

    # Canonical name, beside the shards it came from.
    save_csv(summary, directory, 'variant_summary.csv')

    # The same frame under the name notebook 3 opens. Combined across networks,
    # NOT split: notebook 3 does its own df[df["network"] == ...] and would find
    # a pre-split file the wrong shape.
    save_csv(summary, NOTEBOOK3_DIR, f'{slug}_summary.csv')

    # Per-network convenience copies. Not what notebook 3 reads — these exist so
    # one network can be opened, or handed on, without loading the other two.
    for network in sorted(summary['network'].unique()):
        save_csv(summary[summary['network'] == network], directory,
                 f'variant_summary_{network}.csv')

    summary.insert(0, 'option', slug)
    all_options.append(summary)

if failures:
    (STUDY_ROOT / f'aggregation_failures_{RUN_TAG}.json').write_text(json.dumps(failures, indent=2))
    print(f"\n  {len(failures)} arm(s) failed and were skipped — tracebacks in "
          f"aggregation_failures_{RUN_TAG}.json")

# How complete is this?

In [ ]:
# Written beside the notebook-3 files, because the question "how much of this is
# actually here" travels with them. A summary of a partial arm is a perfectly
# well-formed table — the regressions downstream will fit it and report nothing
# unusual — so the count has to be carried alongside rather than inferred.
COMPLETENESS = pd.DataFrame(completeness)

if len(COMPLETENESS):
    save_csv(COMPLETENESS, NOTEBOOK3_DIR, 'completeness.csv')

    partial = COMPLETENESS[COMPLETENESS['complete'] == False]  # noqa: E712
    if len(partial):
        print(f"\n  {len(partial)} of {len(COMPLETENESS)} arm(s) are PARTIAL:")
        for row in partial.itertuples():
            print(f"    {row.option} / {row.network} / {row.method}: "
                  f"{row.variants_summarised:,} of {row.variants_target:,} ({row.pct}%)")
        print("\n  These aggregated fine and are usable. Re-run the relevant notebook "
              "to bank more\n  variants, then re-run this one — only the new variants "
              "will be read.")
    else:
        print("\n  Every arm is at or above its target.")

COMPLETENESS

# Cross-option tables

In [ ]:
# One frame across every option. `uncertainty` is a per-variant covariate rather
# than a per-option constant, so Option 3's drawn values sit in the same column
# as Options 1 and 2's fixed ones — which is what makes a regression over the
# whole uncertainty range possible from this single file.
#
# The leading `option` column is why this one is NOT a notebook-3 input: it does
# not match the 19-column contract checked below. Use it for cross-option work.
ALL = pd.concat(all_options, ignore_index=True) if all_options else pd.DataFrame()
if len(ALL):
    save_csv(ALL, STUDY_ROOT, f'all_options_variant_summary_{RUN_TAG}.csv')
    print(f"\n  {len(ALL):,} variants across {ALL['option'].nunique()} option(s)")
else:
    print("  Nothing aggregated — no option produced any rows.")

ALL.head()

In [ ]:
# One row per (option, network, arm): how many variants are in, how much of the
# target that is, and where the outcomes sit. The table to read first — it makes
# a mis-slugged option or a half-finished arm obvious before any of it reaches
# an analysis.
if len(ALL):
    aggs = {'variants': ('variant_index', 'size')}
    aggs.update({f'mean_{c}': (f'mean_{c}', 'mean')
                 for c in OUTCOMES if f'mean_{c}' in ALL.columns})
    OVERVIEW = ALL.groupby(['option', 'network', 'method'], as_index=False).agg(**aggs)

    # Carry completeness into the overview so the outcome means are never read
    # without the sample size they were taken over.
    if len(COMPLETENESS):
        OVERVIEW = OVERVIEW.merge(
            COMPLETENESS[['option', 'network', 'method', 'variants_target', 'pct', 'complete']],
            on=['option', 'network', 'method'], how='left')

    save_csv(OVERVIEW, STUDY_ROOT, f'all_options_overview_{RUN_TAG}.csv')
    display(OVERVIEW)
else:
    OVERVIEW = pd.DataFrame()
    print("  Nothing to summarise.")

# Notebook 3 contract check

In [ ]:
# What notebook 3 requires of these files, asserted rather than assumed.
#
# The columns below are the exact header of results/option1_literature_summary.csv
# and results/option3_phase_transition_summary.csv — the files notebook 3 already
# reads. Written out in full, and in order, on purpose: deriving them from the
# module's own constants would agree with summarise_arm by construction and so
# could never catch the failure this guard exists for, which is summarise_arm
# changing out from under a downstream notebook that pins column NAMES
# (`network`, `uncertainty`, `degree_gini_coefficient`,
# `approx_average_clustering_coefficient`,
# `mean_share_of_correct_agents_at_convergence`).
#
# Note what is NOT checked: row counts. A partial arm is a valid input here —
# completeness.csv carries that, and this cell is only about shape.
NOTEBOOK3_COLUMNS = [
    'network', 'method', 'variant_index',
    'proportion_edges', 'uncertainty', 'n_experiments', 'n_agents', 'n_edges',
    'average_degree', 'degree_gini_coefficient',
    'approx_average_clustering_coefficient', 'variation_seed',
    'mean_share_of_correct_agents_at_convergence',
    'std_share_of_correct_agents_at_convergence',
    'mean_convergence_step', 'std_convergence_step',
    'mean_proportion_reached_by_truth', 'std_proportion_reached_by_truth',
    'n_runs',
]

problems = []
written = sorted(p for p in NOTEBOOK3_DIR.glob('*_summary.csv'))
if not written:
    print(f"  Nothing written to {NOTEBOOK3_DIR} — nothing to check.")

for path in written:
    columns = list(pd.read_csv(path, nrows=0).columns)
    if columns == NOTEBOOK3_COLUMNS:
        print(f"  OK  {path.name}")
        continue
    missing = [c for c in NOTEBOOK3_COLUMNS if c not in columns]
    extra = [c for c in columns if c not in NOTEBOOK3_COLUMNS]
    problems.append((path.name, missing, extra))
    print(f"  !!! {path.name}: missing={missing} extra={extra}")

if problems:
    raise AssertionError(
        f"{len(problems)} file(s) do not match the notebook 3 column contract. "
        "Loading them there would fail on a KeyError inside linear_regression(), "
        "which is a long way from the cause — fix it here."
    )

# Also check against the copies already in the repo, when they are reachable.
for slug, title, notebook in OPTIONS:
    reference = REPO_ROOT / 'results' / f'{slug}_summary.csv'
    if reference.exists():
        columns = list(pd.read_csv(reference, nrows=0).columns)
        verdict = 'matches' if columns == NOTEBOOK3_COLUMNS else 'DIFFERS FROM'
        print(f"  results/{reference.name} {verdict} the contract "
              f"({len(columns)} columns)")

# Hand off to notebook 3

In [ ]:
# Notebook 3 reads from the repo's results/ directory, not from Drive, so the
# last step is a copy. Local runs can do it here; Colab runs cannot — a clone in
# /content is discarded with the runtime, and committing from it would be a
# write to the repo this notebook has no business making.
print(f"Files for notebook 3 are in:\n  {NOTEBOOK3_DIR}\n")
for path in sorted(NOTEBOOK3_DIR.iterdir()):
    print(f"  {path.name}")

if RUNNING_LOCALLY:
    destination = REPO_ROOT / 'results'
    destination.mkdir(parents=True, exist_ok=True)
    for path in sorted(NOTEBOOK3_DIR.glob('*_summary.csv')):
        shutil.copy2(path, destination / path.name)
        print(f"  copied -> {destination / path.name}")
else:
    print("\nOn Colab: download that folder from Drive and drop the *_summary.csv "
          "files into\nthe repo's results/ directory, then run "
          "3. Results Data Analysis.ipynb.\nNo rename and no reshape — the "
          "filenames are already the ones it opens.\n\nKeep completeness.csv with "
          "them: a partial arm produces a perfectly ordinary\nregression, and that "
          "file is the only record of how much of it was there.")

# Disconnect from runtime

In [ ]:
# Must be the LAST cell — after every save. `runtime.unassign()` actually frees
# the runtime, whereas `kernel.disconnect()` only detaches the frontend and
# leaves it assigned (and billed). Gated so a developer iterating interactively
# is not kicked out (NOTEBOOK_WRITING_SKILL §11).
#
# Defaults to False here, unlike the run notebooks: this notebook finishes in
# minutes, and its whole purpose is to hand you tables to look at and files to
# download. Releasing the runtime out from under that would be unhelpful.
AUTO_DISCONNECT = False

if AUTO_DISCONNECT and not RUNNING_LOCALLY:
    from datetime import datetime
    import pytz
    stamp = datetime.now(pytz.timezone('America/New_York')).strftime('%Y-%m-%d %H:%M:%S %Z')
    print(f"Finished at {stamp} — releasing runtime.")
    from google.colab import runtime
    runtime.unassign()
else:
    print("Aggregation complete (runtime not released; set AUTO_DISCONNECT=True to free it).")